# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibly one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probability of net winning or losing after 50 rounds?


9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 

In [50]:
class card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def __str__(self):
        return str(self.rank) + " of " + self.suit

In [51]:
card_1 = card("Ace", "Clubs")

print(card_1)
print(card_1.rank)
print(card_1.suit)

Ace of Clubs
Ace
Clubs


In [52]:
import random

class deck:
    def __init__(self, num_decks=6):
        self.cards = list()

        suits = ["Hearts", "Diamonds", "Clubs", "Spades"]
        ranks = ["Ace", "2", "3", "4", "5", "6", "7",
                "8", "9", "10", "Jack", "Queen", "King"]

        for i in range(num_decks):#make 6 complete decks
            for suit in suits:#go through the 4 suits
                for rank in ranks: #go through the 13 cards
                    self.cards.append(card(rank, suit)) # print out the total of cards
        self.needs_shuffle = False
        self.shuffle()
        self.add_plastic_card() 

    def shuffle(self):#code for shuffling cards
        random.shuffle(self.cards)

    def draw(self):#code for drawing cards from a deck pop() allows us to delete the last card so the amount of cards go down
        drawn_card = self.cards.pop()

        if drawn_card == "PLASTIC CARD":
            self.needs_shuffle = True
            drawn_card = self.cards.pop()

        return drawn_card

    def add_plastic_card(self):
        position = random.randint(1, len(self.cards) -1)
        self.cards.insert(position, "PLASTIC CARD")

In [53]:
class hand:
    def __init__(self):
        self.cards = list()

    def add_card(self, new_card): #function for when a new card is being received from the deck
        self.cards.append(new_card)#add new card to the end of the hand

    def get_value(self): #func for getting total blackjack value
        total = 0 

        aces = 0

        for current_card in self.cards: #card values check for values greater than 10
            if current_card.rank in ["Jack", "Queen", "King"]:
                total += 10

            elif current_card.rank == "Ace": #verification to see if its an ace

                total += 11
                aces += 1

            else: #if not convert the # card into an int
                total += int(current_card.rank)

        while total > 21 and aces > 0:
            total -= 10
            aces -= 1

        return total

In [54]:
class player:

    def __init__(self, name, chips = 100):# gives each player 100 chips
        self.name = name
        self.chips = chips

        #create a new hand for player
        self.hand = hand()

        self.bet = 0

    def hit(self, deck): #to get another card

        new_card = deck.draw() #draw one card from the deck.

        self.hand.add_card(new_card) #add that card to player hand


    def stay(self):#func for passing on receving card
        print(self.name, "has decided to stay.")


    def place_bet(self, amount):
        if amount > 0 and amount <= self.chips: #verifciation to make sure bet is positve and player has enough to bet
             
             self.chips -= amount #subtract the amount bet from chips

             self.bet = amount #store how much player bet

        else:
            print("Your bet is invalid!")


In [55]:
print(hasattr(player, "hit"))

True


In [56]:
print(hasattr(player, "hit"))
print(player.__dict__.keys())

True
dict_keys(['__module__', '__init__', 'hit', 'stay', 'place_bet', '__dict__', '__weakref__', '__doc__'])


In [57]:
class dealer(player):
    
    def __init__(self):

        super().__init__("Dealer", 0)#setting dealer name to Dealer and reseting chips to 0

    def take_turn(self,deck): #function for dealers action

        while self.hand.get_value() < 17:

            self.hit(deck)

            self.stay()

In [58]:
class game:

    def __init__(self, num_players=1):

        self.deck = deck() #creation of a new deck from deck class above

        self.players = list()

        for i in range(num_players):

            self.players.append(player("Player " + str(i + 1))) #Creation of a player and giving each player a diff num

        self.dealer = dealer()

    def play_round(self):

        if self.deck.needs_shuffle: #make a new deck if the plastic card was hit
            self.deck = deck()

        current_player = self.players[0] #code for selecting first player

        current_player.hand = hand()

        self.dealer.hand = hand()

        
        # Ask for a bet
        bet = int(input("Bet: "))
        if bet <= 0 or bet > current_player.chips:
            print("Invalid bet")
            return

        current_player.place_bet(bet)

        # Deal two cards each
        for i in range(2):
            current_player.hit(self.deck)
            self.dealer.hit(self.deck)

        print("Your cards:", [str(c) for c in current_player.hand.cards])
        print("Dealer shows:", self.dealer.hand.cards[0])

        # Player's turn
        while current_player.hand.get_value() < 21:
            choice = input("Hit or stay? ").lower()

            if choice == "hit":
                current_player.hit(self.deck)
                print("Your total:", current_player.hand.get_value())
            elif choice == "stay":
                break

        # Dealer's turn
        if current_player.hand.get_value() <= 21:
            self.dealer.take_turn(self.deck)

        # Get both totals
        a = current_player.hand.get_value()
        b = self.dealer.hand.get_value()

        print("Your total:", a, "Dealer total:", b)

        # Decide the winner
        if a > 21:
            print("You lose")
        elif b > 21 or a > b:
            print("You win")
            current_player.chips += bet * 2
        elif a == b:
            print("Tie")
            current_player.chips += bet
        else:
            print("You lose")

        current_player.bet = 0
        print("Chips:", current_player.chips)




so for the above code i had ran into some coding errors like indention errors and spelling mistakes that i had chat clear up

In [74]:
class computer_player(player):

    def take_turn(self,deck):
        while self.hand.get_value() < 17:
            self.hit(deck)
        self.stay()

In [60]:
deck_1 = deck()
print(len(deck_1.cards))

313


In [61]:
deck_1 = deck()

print("Before drawing:", len(deck_1.cards))

deck_1.shuffle()

card_1 = deck_1.draw()

print("Card drawn is:", card_1)
print("After drawing cards:", len(deck_1.cards))


Before drawing: 313
Card drawn is: King of Clubs
After drawing cards: 312


had an issue running it at first it was giving me an error telling me that i had a list and i had chat help me figure out what was pringting wrong for the check

In [62]:
deck_1 = deck()

print("Total amount of items:", len(deck_1.cards))
print("Need to Shuffle:", deck_1.needs_shuffle)

for i in range(312):
    deck_1.draw()

print("Needs shuffling after drawing a new card:", deck_1.needs_shuffle)

Total amount of items: 313
Need to Shuffle: False
Needs shuffling after drawing a new card: True


Task 3 test

In [63]:
hand_1 = hand()

print(hand_1.cards)
print(len(hand_1.cards))

[]
0


In [64]:
#testing to see if it works
player_1 = player("Bryant")


#player's info
print("Name:", player_1.name)
print("Chips:", player_1.chips)
print("Hand:", player_1.hand.cards)
print("Bet:",player_1.bet)

Name: Bryant
Chips: 100
Hand: []
Bet: 0


In [65]:
dealer_1 = dealer()

print("Name:", dealer_1.name)
print("Chips:", dealer_1.chips)
print("Hands:", dealer_1.hand.cards)
print("Bet:", dealer_1.bet)


Name: Dealer
Chips: 0
Hands: []
Bet: 0


In [66]:
game_1 = game(4)

print("Number of player in this game:", len(game_1.players))

print("Dealer:", game_1.dealer.name)

for p in game_1.players:
    print(p.name, "- Chip amount:", p.chips)

Number of player in this game: 4
Dealer: Dealer
Player 1 - Chip amount: 100
Player 2 - Chip amount: 100
Player 3 - Chip amount: 100
Player 4 - Chip amount: 100


Task 4 testing code

In [67]:
hand_1 = hand()

card_1 = card("King", "Clubs")
card_2 = card("Queen", "Diamonds")

#code for addiing to the hand

hand_1.add_card(card_1)
hand_1.add_card(card_2)

for c in hand_1.cards:
    print(c)

King of Clubs
Queen of Diamonds


In [68]:
hand_1 = hand()

hand_1.add_card(card("Ace", "Diamonds"))
hand_1.add_card(card("Queen", "Hearts"))

print(hand_1.get_value())

21


In [69]:
deck_1 = deck()
player_1 = player("Ant")

print("Before hit:", len(player_1.hand.cards))

player_1.hit(deck_1)

print("After hit:", len(player_1.hand.cards))

for current_card in player_1.hand.cards:
    print(current_card)

print("Hand value:", player_1.hand.get_value())

#when i was running this test cell for the hits i was running into a lot of errors such as naming and indention that i had chat clear up and explain

Before hit: 0
After hit: 1
3 of Clubs
Hand value: 3


In [70]:
print(player.__dict__.keys())

dict_keys(['__module__', '__init__', 'hit', 'stay', 'place_bet', '__dict__', '__weakref__', '__doc__'])


In [71]:
player_1 = player("Ant")

print("Starting chip amount:", player_1.chips)

player_1.place_bet(30)

print("Remaining chip amount is:", player_1.chips)
print("Current bet:", player_1.bet)

player_1.stay()

Starting chip amount: 100
Remaining chip amount is: 70
Current bet: 30
Ant has decided to stay.


In [73]:
game_1 = game()
game_1.play_round()

Your cards: ['2 of Hearts', '2 of Hearts']
Dealer shows: Queen of Diamonds
Dealer has decided to stay.
Your total: 4 Dealer total: 24
You win
Chips: 120


Task 5 testing

In [75]:
# Test the computer players
d = deck()
dealer_1 = dealer()

players = list()

for i in range(3):
    players.append(computer_player("Player " + str(i + 1)))

for round_number in range(3):
    print("\nRound", round_number + 1)

    dealer_1.hand = hand()

    for p in players:
        p.hand = hand()
        p.hit(d)
        p.hit(d)

    dealer_1.hit(d)
    dealer_1.hit(d)

    for p in players:
        p.take_turn(d)

    dealer_1.take_turn(d)

    print("Dealer:", dealer_1.hand.get_value())

    for p in players:
        print(p.name, p.hand.get_value())

    if d.needs_shuffle:
        d = deck()


Round 1
Player 1 has decided to stay.
Player 2 has decided to stay.
Player 3 has decided to stay.
Dealer: 21
Player 1 19
Player 2 21
Player 3 17

Round 2
Player 1 has decided to stay.
Player 2 has decided to stay.
Player 3 has decided to stay.
Dealer has decided to stay.
Dealer: 18
Player 1 21
Player 2 19
Player 3 18

Round 3
Player 1 has decided to stay.
Player 2 has decided to stay.
Player 3 has decided to stay.
Dealer has decided to stay.
Dealer: 17
Player 1 19
Player 2 19
Player 3 26
